In [18]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath(os.path.join('..')))

In [1]:
import os
import sys

# 1. Force install libraries inside the active notebook kernel path
!{sys.executable} -m pip install xgboost shap

# 2. Shift python path up one level to see the project root 'src/' directory
sys.path.append(os.path.abspath(os.path.join('..')))

print("✅ [SYSTEM] Dependencies installed and path mappings configured successfully.")

✅ [SYSTEM] Dependencies installed and path mappings configured successfully.


In [6]:
import pandas as pd
from src.modeling import train_risk_models, calculate_risk_premium
from src.interpretability import generate_model_explanations

# Load processed DVC data repository reference
data_path = '../data/processed/cleaned_insurance_data.csv' 
df = pd.read_csv(data_path)

print(f"📊 Dataset successfully loaded. Shape: {df.shape}")

📊 Dataset successfully loaded. Shape: (10000, 21)


In [9]:
# Train severity (regression) and probability (classification) engines
# The modeling package automatically filters columns and extracts continuous features
severity_model, probability_model, feature_cols = train_risk_models(df)

print("\n✨ Selected Predictor Features Used for Training Matrix:")
print(feature_cols)

🚀 [SYSTEM] Initializing Task 4 Statistical Modeling Loop...

📊 Running Severity Benchmarking (Subset where PastClaims > 0)...
   -> Linear Regression | RMSE: 0.62, R2 Score: 0.4594
   -> Random Forest | RMSE: 0.56, R2 Score: 0.5631
   -> XGBoost Regressor | RMSE: 0.61, R2 Score: 0.4734

📊 Running Probability Benchmarking...
   -> Logistic Regression | LogLoss Score: 0.3886, Accuracy: 0.8490
   -> Random Forest Classifier | LogLoss Score: 0.2895, Accuracy: 0.9025
   -> XGBoost Classifier | LogLoss Score: 0.2898, Accuracy: 0.8930

✨ Selected Predictor Features Used for Training Matrix:
['Age', 'AnnualIncome', 'RiskScore', 'Deductible', 'NCD', 'ClaimAmount', 'TotalPremium', 'TotalClaims', 'CustomValueEstimate', 'ZipCode', 'VehicleAge']


In [22]:
import numpy as np
import pandas as pd

def local_prepare_modeling_features(df):
    """Dynamically locates your columns and prepares predictors locally."""
    df_clean = df.copy()
    
    # Dynamic column finding matching your original Task 2 setup
    premium_col = [c for c in df_clean.columns if 'premium' in c.lower()][0]
    claim_col = [c for c in df_clean.columns if 'claim' in c.lower()][0]
    
    # Feature Engineering Layer: Convert Registration Years to an active Age metric
    reg_year_opts = [c for c in df_clean.columns if 'year' in c.lower() or 'reg' in c.lower()]
    if reg_year_opts:
        df_clean['VehicleAge'] = 2026 - df_clean[reg_year_opts[0]]
    else:
        df_clean['VehicleAge'] = 0
            
    return df_clean, claim_col


def local_calculate_risk_premium(df, severity_model, probability_model, features, expense_loading=150.0, profit_margin=0.15):
    """Calculates optimized annual premiums calibrated to match the true economic scale."""
    # Call the locally defined preparation script
    df_proc, _ = local_prepare_modeling_features(df)
    X = df_proc[features]
    
    # 1. Generate core statistical predictions from your pipeline
    p_claim = probability_model.predict_proba(X)[:, 1]
    pred_severity = severity_model.predict(X)
    
    # 2. Prevent negative severity predictions from clipping values down
    pred_severity = np.clip(pred_severity, a_min=1.0, a_max=None)
    
    # 3. Calculate raw monthly or baseline risk exposure
    raw_pure_premium = p_claim * pred_severity
    
    # 4. DYNAMIC ACTUARIAL SCALING (Burning Cost Method)
    old_premium_col = [c for c in df.columns if 'premium' in c.lower()][0]
    target_average_premium = df[old_premium_col].mean()
    
    # Scale factor pushes the mean of the raw risk premiums up to realistic levels
    scale_factor = (target_average_premium * (1 - profit_margin) - expense_loading) / (raw_pure_premium.mean() + 1e-6)
    
    # If models are completely zero-inflated, safeguard with a default pricing multiplier
    if scale_factor <= 0 or np.isnan(scale_factor):
        scale_factor = 120.0  
        
    # 5. Compute the fully calibrated premium vector
    calibrated_pure_premium = raw_pure_premium * scale_factor
    optimized_premiums = (calibrated_pure_premium + expense_loading) / (1 - profit_margin)
    
    # 6. Apply clipping thresholds to mirror standard market boundaries
    min_historical = df[old_premium_col].min()
    max_historical = df[old_premium_col].max()
    df_proc['Calculated_Risk_Premium'] = np.clip(optimized_premiums, min_historical, max_historical)
    
    print(f"✅ [SUCCESS] Dynamic scaling factor applied: {scale_factor:.4f}")
    return df_proc


# --- EXECUTE THE LOCAL FIX ---
# Run the pricing calculation using our local, self-contained functions
priced_df = local_calculate_risk_premium(df, severity_model, probability_model, feature_cols)

# Discover the historical premium column dynamically to print the comparison
old_premium_col = [c for c in df.columns if 'premium' in c.lower()][0]

print("\n📋 Comparison Sample: Historical Premium vs. Newly Calibrated Risk Premium:")
print(priced_df[[old_premium_col, 'Calculated_Risk_Premium']].head(10))

✅ [SUCCESS] Dynamic scaling factor applied: 1827.5972

📋 Comparison Sample: Historical Premium vs. Newly Calibrated Risk Premium:
   AnnualPremium  Calculated_Risk_Premium
0           2346              2578.699389
1           2334              5105.000000
2           1697              2248.412649
3           2370               951.000000
4           2582              5105.000000
5           1310              2317.713825
6           2204               951.000000
7           1590               951.000000
8           2665              2325.017225
9           2527              5105.000000


In [23]:
# Pass the champion severity engine to generate feature impact visualizations
generate_model_explanations(severity_model, priced_df, feature_cols)


🚀 [SYSTEM] Executing Model Interpretability Framework (SHAP Engine)...
[SUCCESS] Exported SHAP explainability charts to: ../reports/figures/05_shap_feature_importance.png
